# За вкъщи: упражнение 3, част 1

**Данни за модел, който играе**

1. Пишете кода в съответните клетки на тази тетрадка.
2. Преди предаване рестартирайте ядрото и изпълнете цялата тетрадка.
3. Изпратете я на mkorudzhiev@uni-sofia.bg с тема "Уп. 3, част 1 - <Вашето име>, <Фак. номер>".

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# Играта и 12-те известни стратегии, както в упражнението
POINTS = {"CC": 3, "CD": 0, "DC": 5, "DD": 1} # ходът на играча и ходът на другия
ROUNDS = 100 # рундове в един мач
rng = np.random.default_rng(42) # генераторът на случайни числа за всички мачове


def always_c(my, their):
    return "C"


def always_d(my, their):
    return "D"


def random_move(my, their):
    return "C" if rng.random() < 0.5 else "D"


def tit_for_tat(my, their):
    return their[-1] if their else "C"


def suspicious_tft(my, their):
    return their[-1] if their else "D"


def tit_for_two_tats(my, their):
    return "D" if their[-2:] == ["D", "D"] else "C"


def grudger(my, their):
    return "D" if "D" in their else "C"


def pavlov(my, their):
    if not my:
        return "C"
    if their[-1] == "C":
        return my[-1]
    return "D" if my[-1] == "C" else "C"


def joss(my, their):
    move = tit_for_tat(my, their)
    return "D" if move == "C" and rng.random() < 0.1 else move


def alternator(my, their):
    return "C" if len(my) % 2 == 0 else "D"


def hard_tft(my, their):
    return "D" if "D" in their[-3:] else "C"


def detective(my, their):
    if len(my) < 4:
        return "CDCC"[len(my)]
    return their[-1] if "D" in their[:4] else "D"


STRATEGIES = {"Always C": always_c, "Always D": always_d, "Random": random_move,
              "Tit for Tat": tit_for_tat, "Suspicious TFT": suspicious_tft,
              "Tit for Two Tats": tit_for_two_tats, "Grudger": grudger, "Pavlov": pavlov,
              "Joss": joss, "Alternator": alternator, "Hard TFT": hard_tft, "Detective": detective}


def play(strategy_a, strategy_b, rounds=ROUNDS):
    """Мач между две стратегии. Връща ходовете на двете като низове."""
    moves_a, moves_b = [], []
    for _ in range(rounds):
        a = strategy_a(moves_a, moves_b)
        b = strategy_b(moves_b, moves_a)
        moves_a.append(a)
        moves_b.append(b)
    return "".join(moves_a), "".join(moves_b)


def points(moves, opp_moves):
    """Точките на първия играч в мача."""
    return sum(POINTS[a + b] for a, b in zip(moves, opp_moves))


def tournament(players, opponents, matches=1):
    """Всеки играч срещу всеки противник. Един ред на мач."""
    rows = []
    for name, strategy in players.items():
        for opp_name, opponent in opponents.items():
            for match in range(1, matches + 1):
                moves, opp_moves = play(strategy, opponent)
                rows.append({"player": name, "opponent": opp_name, "match": match,
                             "moves": moves, "opp_moves": opp_moves})
    return pd.DataFrame(rows)


print("Известни стратегии:", len(STRATEGIES))

In [ ]:
# Примерите, моделът и проверката от упражнението
SITUATIONS = ["начало", "CC", "CD", "DC", "DD"]


def decisions(games, horizon=5):
    """Един ред на решение: ситуация, ход и етикет от horizon рунда."""
    rounds = games.assign(move=games["moves"].map(list), opp_move=games["opp_moves"].map(list))
    rounds = rounds.explode(["move", "opp_move"])
    by_match = rounds.groupby(level=0)
    rounds["round"] = by_match.cumcount() + 1
    rounds["situation"] = (by_match["move"].shift() + by_match["opp_move"].shift()).fillna("начало")
    rounds["points"] = (rounds["move"] + rounds["opp_move"]).map(POINTS)
    by_match = rounds.groupby(level=0)["points"]
    rounds["future"] = sum(by_match.shift(-k) for k in range(horizon)) # този и следващите horizon - 1 рунда
    columns = ["player", "opponent", "match", "round", "situation", "move", "future"]
    return rounds.dropna(subset=["future"])[columns].reset_index(drop=True)


def learn(data):
    """Правилото на модела: във всяка ситуация ходът с по-голям среден етикет."""
    averages = data.pivot_table(index="situation", columns="move", values="future", aggfunc="mean")
    return averages.idxmax(axis=1).to_dict()


def from_rule(rule):
    """Стратегия, която във всяка ситуация играе хода от правилото."""
    def strategy(my, their):
        situation = my[-1] + their[-1] if my else "начало"
        return rule[situation]
    return strategy


def tournament_score(strategy, matches=30):
    """Средни точки на ход срещу всяка от 12-те известни стратегии."""
    trial = tournament({"проба": strategy}, STRATEGIES, matches=matches)
    trial["score"] = [points(a, b) / ROUNDS for a, b in zip(trial["moves"], trial["opp_moves"])]
    return trial.groupby("opponent")["score"].mean()


# Същите записи като data/ipd_random_games.csv: Random срещу 12-те стратегии, по 200 мача
rng = np.random.default_rng(2027)
records = tournament({"Random": random_move}, STRATEGIES, matches=200)
data = decisions(records)
rule = learn(data)
print("Мачове:", len(records), " примери:", len(data))
print("Правилото от упражнението:", {s: rule[s] for s in SITUATIONS})

rng = np.random.default_rng(7)
print(f"Точки на ход в проверката: {tournament_score(from_rule(rule)).mean():.3f}")

## Задача 1. Данни само срещу един противник

В упражнението моделът учи от мачовете срещу всичките 12 стратегии наведнъж.
Какво научава, ако данните са само срещу един противник?

1. За всеки противник оставете от `data` само редовете срещу него и научете
   правило с `learn`. Проверете това правило в 30 мача срещу същия
   противник. Проверете по същия начин и правилото `rule` от упражнението.
   Срещу кои противници правилото, научено само срещу тях, събира повече
   точки?
2. Какво правило се научава срещу Always C и срещу Tit for Two Tats? Защо
   срещу Always C в правилото липсват ситуациите CD и DD?
3. Защо тогава в упражнението моделът не учи отделно правило за всеки
   противник?

<details><summary>Подсказка</summary>

Мачовете срещу един противник са `tournament({"правило": from_rule(own)},
{name: opponent}, matches=30)`. Средните точки на ход са
`np.mean([points(a, b) for a, b in zip(trial["moves"], trial["opp_moves"])]) / ROUNDS`.
Задавайте `rng = np.random.default_rng(7)` преди всяка проверка, за да
играят двете правила при едни и същи условия.

</details>

In [ ]:
# rows = []
# for name, opponent in STRATEGIES.items():
#     own = learn(data[data["opponent"] == name])
#     ...
# pd.DataFrame(rows)

*Отговор за задача 1:*

## Задача 2. Нечестна монета

В упражнението Random хвърля честна монета: C и D имат вероятност 0.5.
Какво научава моделът, ако монетата е нечестна?

1. Напишете стратегия `lazy_coin`, която играе C с вероятност 0.2 и D с
   вероятност 0.8, по образеца на `random_move`. Съберете данни както в
   упражнението: `rng = np.random.default_rng(2027)`, по 200 мача срещу всеки
   противник, после `decisions`.
2. Колко примера има за всяка ситуация и ход? Сравнете с данните от честната
   монета. Кои ситуации стават редки и защо?
3. Научете правило и го проверете с `tournament_score` (семе 7). С какво се
   различава от правилото `rule` и колко точки събира?
4. Етикетът са точките в този и в следващите 4 рунда. Кой избира ходовете в
   тези 4 рунда? Защо това променя наученото правило?

<details><summary>Подсказка</summary>

Броят на примерите е `pivot_table(index="situation", columns="move",
values="future", aggfunc="size")`.

</details>

In [ ]:
# def lazy_coin(my, their):
#     return ...
#
# rng = np.random.default_rng(2027)
# lazy_records = tournament({"Lazy coin": lazy_coin}, STRATEGIES, matches=200)
# lazy_data = decisions(lazy_records)
# ...

*Отговор за задача 2:*

## Задача 3. Памет от два рунда

В упражнението ситуацията е само предишният рунд. Какво научава моделът,
ако вижда два рунда назад?

1. Напишете `decisions2`, като `decisions`, но ситуацията да са ходовете в
   двата предишни рунда, например "DCCC": преди два рунда играчът е играл D, а
   другият C, а в предишния рунд и двамата са играли C. Във втория рунд има
   само един предишен рунд, затова там ситуацията е от две букви, а в първия
   е "начало". Колко ситуации има? Колко е най-малкият брой примери за
   ситуация и ход и в коя ситуация е?
2. Научете правило с `learn` и го превърнете в стратегия с функцията
   `from_rule2` от клетката по-долу. Колко точки събира с `tournament_score`
   (семе 7)? Сравнете с правилото `rule`, общо и срещу всеки противник.
3. Защо повече памет не помага тук?

<details><summary>Подсказка</summary>

Колона `pair` с двата хода от всеки рунд улеснява работата:
`by_match["pair"].shift(2).fillna("") + by_match["pair"].shift(1)` дава
ситуацията, а за първия рунд остава `fillna("начало")`.

</details>

In [ ]:
# def decisions2(games, horizon=5):
#     """Като decisions, но ситуацията са ходовете в двата предишни рунда."""
#     rounds = games.assign(move=games["moves"].map(list), opp_move=games["opp_moves"].map(list))
#     rounds = rounds.explode(["move", "opp_move"])
#     rounds["pair"] = ...
#     ...
#
#
# def from_rule2(rule):
#     """Стратегия, която гледа ходовете в двата предишни рунда."""
#     def strategy(my, their):
#         pairs = [a + b for a, b in zip(my[-2:], their[-2:])]
#         return rule["".join(pairs)] if pairs else rule["начало"]
#     return strategy

*Отговор за задача 3:*